In [83]:
import os
import time
import shutil
import numpy as np
import re
import scipy.special 
from numpy import linalg as LA


# GIVE THE PARAMETERS
m = 301 #first monomer
n = 302 #last monomer
total_s = 5 #number of computed roots
s = 5 # number of states considered in FRENKEL e.g. 2 states for Qy band
key = "VNN_AB"


# COMPUTE SOME STUFF
A = n - m + 1 # number of molecules 
B = n - m #number of molecules minus one 
number_coup = int(scipy.special.comb(total_s + total_s, 2))

In [84]:
"""
variable that collects the coupling terms. Ordering:
  C = [Mol_i][Mol_i_state_start][Mol_i_state_end][Mol_j][Mol_j_state_start][Mol_j_state_end]
Mol_i goes from 0 to A-1
"""
C = np.zeros((A, s, s, A, s, s))

"""
variable that collects the energy terms. Ordering:
  E = [Mol_i][Mol_i_state]
Mol_i goes from 0 to A-1
"""
E = np.zeros((A, s))

"""
variable that collects the transition dipole moments (TDM) to compute spectroscopic properties. Ordering:
  TDM = [Mol_i, Mol_i_state_start, Mol_i_state_end, space components (x,y,z)]
"""
TDM = np.zeros((A,total_s, total_s, 3))
TOT_TDM = int(scipy.special.comb(total_s, 2))

In [85]:
C.shape

(2, 5, 5, 2, 5, 5)

In [86]:
with open("coupling_301_302.out", "r") as F:
    output = F.readlines()

In [87]:
for i in range(m, n+1): #m = 301; n+1 = 328
 for j in range(i+1, n+1): 		# i+1 = 302; n+1 = 328. consider the dimeric interaction in an upper-trinagular fashion.
# define the destination directory for the files
#  dest = os.path.join(source,'coup_'+alfa+'_'+beta)
# define the path for the output file; coup_'+alfa+'_'+beta is the specific name given to the files and folders.
# open the output file
  with open("coupling_301_302.out", "r") as F:
   output = F.readlines()

 #COUPLING TERMS COLLECTION
 
#find the line number of the keyword 
  iline = 0
  for k  in range(len(output)-1,0, -1): 	#goes from the bottom to the top 
   try:
    if output[k].split()[0] == key:
     iline = k					#iline is placed at the key word.
     break
   except:
    pass
  for l in range(1, number_coup + 1):	
   line = output[iline + l].split() 	#takes the iline and splits it: ['<(001)B(001)A|v|(001)B(001)A>', '=', '-0.10072418E-01']
   indeces = re.split('\(|\)', line[0]) #takes the first space and split it at brakets: ['<', '001', 'B', '001', 'A|v|', '001', 'B', '001', 'A>']
   A_start = int(indeces[7])
   A_end = int(indeces[3])
   B_start = int(indeces[5])
   B_end = int(indeces[1])
   if (A_start <= s) and (A_end <= s) and (B_start <= s) and (B_end <= s):
    C[i - m][A_start - 1][A_end - 1][j - m][B_start - 1][B_end - 1] = float(line[2]) # upper triangular; the monomers goes from 0(301) to 26(327), same for states (from 0 to s-1)
    
  #ENERGY COLLECTION
  
  if i == m: #collect the energies only from the interaction with the first monomer
   
   Esa = 0
   Esb = 0
   for q in range(iline + number_coup + 1, len(output)):
    try:
     if output[q].split()[0] == "states":
      if output[q].split()[1] == "1":
       if Esb < s:
        E[j-m][Esb] = float(output[q].split()[2])		#collect the state (from 0 to s-1) energy for the B monomer (from 1 to 26)
        Esb += 1
      elif  (output[q].split()[1] == "2") and (j == m + 1):	#collect the state (from 0 to s-1) energy for the first monomer only in the first interaction file
       if Esa < s:
        E[i - m][Esa] = float(output[q].split()[2])
        Esa += 1
     elif output[q].split()[0] == "add":
      iline = q							#update the iline position after the states energy list
      break
    except:
     pass
   #end "for q"
	  
  # TDM COLLECTION: note it is upper triangular
   jline = 0
   for q in range(iline, len(output)):  
    try:
     if output[q].split()[0] == "Dip1:":
      jline = q
      break
    except:
     pass  
   TDM_s_start = 0
   TDM_s_end = 1
   for q in range(jline + 1, len(output)): 		#scroll downside collecting the TDM components for the B monomer (j - m)
    if output[q].split()[0] == "Dip2:":			#TDM for first monomer that we want to collect just once
     jline = q
     TDM_s_start = 0
     TDM_s_end = 1
     break
    else:						#collection of the TDM for the m+1 monomers for all the states considered in RASSI
     x_comp = output[q].split()[0]
     y_comp = output[q].split()[1]
     z_comp = output[q].split()[2]
     TDM[j-m][TDM_s_start][TDM_s_end][0] = float(x_comp)
     TDM[j-m][TDM_s_start][TDM_s_end][1] = float(y_comp)
     TDM[j-m][TDM_s_start][TDM_s_end][2] = float(z_comp)
     TDM_s_end += 1
     if TDM_s_end == total_s: 
      TDM_s_start += 1
      TDM_s_end = TDM_s_start + 1
   #end for cycle
   if (j == m + 1):					#collection of the TDM components for m (301) monomer just once
    for q in range(jline + 1, jline + 1 + TOT_TDM):
      x_comp = output[q].split()[0]
      y_comp = output[q].split()[1]
      z_comp = output[q].split()[2]
      TDM[i-m][TDM_s_start][TDM_s_end][0] = float(x_comp)
      TDM[i-m][TDM_s_start][TDM_s_end][1] = float(y_comp)
      TDM[i-m][TDM_s_start][TDM_s_end][2] = float(z_comp)
      TDM_s_end += 1
      if TDM_s_end == total_s:
       TDM_s_start += 1
       TDM_s_end = TDM_s_start + 1


<>:23: SyntaxWarning: invalid escape sequence '\('
<>:23: SyntaxWarning: invalid escape sequence '\('
/var/folders/5b/8k2_sj653zj2b6k3x_0m6tyr0000gn/T/ipykernel_65436/2998176183.py:23: SyntaxWarning: invalid escape sequence '\('
  indeces = re.split('\(|\)', line[0]) #takes the first space and split it at brakets: ['<', '001', 'B', '001', 'A|v|', '001', 'B', '001', 'A>']


In [88]:

###### END OF COLLECTION SECTION ######################
#  C and TDM are upper triangular matrices

#aggregate ground state
#Eg0 = 0.0
#for i in range(0, A):
# Eg0 += E[i][0]			#sum all the ground states of the monomers
# for j in range(i, A):
Eg0 = sum(E[i][0] for i in range(A))
"""
  sum all the couplings gg,gg between monomer i and all the other j-th,
  since C is upper triangular, there isn't double counting
"""
#Eg0 += C[i][0][0][j][0][0]
Eg0 += sum(C[i][0][0][j][0][0] for i in range(A) for j in range(i, A))


# Helper function for computing diagonal energy terms
def compute_diagonal_energy(ENA, agg_s):
    energy = Eg0 - E[ENA][0] + E[ENA][agg_s]
    for M in range(A):
        energy += (
            C[ENA][agg_s][agg_s][M][0][0] + C[M][0][0][ENA][agg_s][agg_s]
            - C[ENA][0][0][M][0][0] - C[M][0][0][ENA][0][0]
 # we want double counting: C[ENA][agg_s][agg_s][M][0][0] + C[M][0][0][ENA][agg_s][agg_s]; 
 # because the Cmn wants n higher than m otherwise it's zero.
        )
    return energy

############ FRENKEL-DAVYDOV #############

#number of aggregate states
N = (A * (s -1)) + 1

#Frankel matrix 
#F[A, states of A, B, state of B]
F = np.zeros(( N, N ))

#DIAGONAL ASSIST-VARIABLE
agg_s = 1
ENA = 0 	#this is a variable that counts the excited monomer in the diagonal terms
agg_s_end = 1
CA = 0		#
diff = 2
a = range(0, A)
p = 0
w = 1
sss_start = 1
sss_end = 1




#new variables
ENA = 0
agg_s = 1
CA = 0
agg_s_end = 1


for i in range(0, N ):
 for j in range(0, N ):
  if i == j: 		#DIAGONAL TERMS 
   if i == 0:		# here we have just the energy of the aggregate ground state Eg0
    F[i][j] = Eg0
   else:			
    """
    here we have the excitation of the ENA-th monomer, each time i == j, agg_s increases i.e. the excitation of the same monomer to the agg_s-th state.
    when agg_s == s, we change monomer ENA += 1 and agg_s in re-initialised to 1
    """
    F[i][j] = Eg0 - E[ENA][0] + E[ENA][agg_s]
    """
    Add all the coupling terms eg|eg between the ENA-th monomer and all the other monomers, removing the gg|gg terms for each M
    """
    for M in range(0, A):
     F[i][j] +=  C[ENA][agg_s][agg_s][M][0][0] + C[M][0][0][ENA][agg_s][agg_s] - C[ENA][0][0][M][0][0] - C[M][0][0][ENA][0][0]
     # we want double counting: C[ENA][agg_s][agg_s][M][0][0] + C[M][0][0][ENA][agg_s][agg_s]; because the Cmn wants n higher than m otherwise it's zero.    
    agg_s += 1
    if agg_s >= s:
     agg_s = 1
     ENA += 1
  #end of diagonal terms  
  
  elif (i == 0) and (j > 0): #FIRST ROW OFF DIAGONAL TERMS, ground state excitations
    F[i][j]  = sum(
       C[CA][0][agg_s_end][M][0][0] + C[M][0][0][CA][0][agg_s_end] for M in range(A)
   )
    agg_s_end += 1
    if agg_s_end >= s:
     agg_s_end = 1
     CA += 1
     if CA >= A:
      CA = 0
      agg_s_end = 1
  #end if first row

  elif (i > 0) and (j > i):
   p = (i - 1) // (s - 1)
   sss_start = (i - 1) % (s - 1) + 1
   w = (j - 1) // (s - 1)
   sss_end = (j - 1) % (s - 1) + 1
   if p == w:  # Intramolecular coupling within the same molecule
    F[i][j] = sum(C[p][sss_start][sss_end][i][0][0] for i in range(A)) +  sum(C[i][0][0][p][sss_start][sss_end] for i in range(A))
   else:  # Intermolecular coupling between different molecules
    F[i][j] = C[p][sss_start][0][w][0][sss_end]

#END FRENKEL MATRIX it is in Hartree 

In [89]:
F.shape[0]

9

In [90]:
F_del = (F + F.T - np.diag(np.diag(F))) - (np.diag(np.ones(N)*Eg0))

In [91]:
F_del * 27.2114

array([[ 0.00000000e+00, -5.58882863e-03,  3.76466753e-02,
        -1.02150268e-02, -1.90080462e-02,  1.07904850e-02,
         3.94887048e-02,  2.28322063e-03,  9.83222632e-04],
       [-5.58882863e-03,  1.62208796e+00, -7.01915750e-03,
        -1.93218666e-02, -1.64585783e-02, -4.76108886e-02,
         9.61676291e-03, -4.45647057e-03, -7.02018364e-05],
       [ 3.76466753e-02, -7.01915750e-03,  2.36422861e+00,
        -5.17778546e-03,  2.66463926e-03,  6.35232718e-03,
         1.47043698e-02,  1.09765666e-03, -7.56638801e-03],
       [-1.02150268e-02, -1.93218666e-02, -5.17778546e-03,
         2.60225017e+00, -2.70014978e-04,  1.13433094e-03,
         6.12146022e-04, -7.90216253e-04, -3.67159012e-04],
       [-1.90080462e-02, -1.64585783e-02,  2.66463926e-03,
        -2.70014978e-04,  2.98953442e+00,  3.07216461e-03,
        -2.54878370e-03,  4.65489365e-04,  1.89778263e-03],
       [ 1.07904850e-02, -4.76108886e-02,  6.35232718e-03,
         1.13433094e-03,  3.07216461e-03,  1.602524

In [92]:
print('\n'.join(['\t'.join(['{:4}'.format(item) for item in row]) 
      for row in F_del*27.2114]))



 0.0	-0.005588828627384	0.03764667529574	-0.010215026768368	-0.019008046182044	0.010790485020034001	0.039488704759360005	0.0022832206267552	0.0009832226317158
-0.005588828627384	1.6220879602904086	-0.00701915750031	-0.019321866550596003	-0.016458578278706	-0.04761088860380001	0.009616762914476001	-0.004456470570966	-7.020183642204e-05
0.03764667529574	-0.00701915750031	2.3642286107985564	-0.005177785464114	0.0026646392558618004	0.00635232717704	0.01470436979899	0.0010976566641452	-0.007566388006186
-0.010215026768368	-0.019321866550596003	-0.005177785464114	2.602250165317007	-0.00027001497791136	0.0011343309373408	0.000612146021716	-0.0007902162532258001	-0.0003671590119532
-0.019008046182044	-0.016458578278706	0.0026646392558618004	-0.00027001497791136	2.9895344225796014	0.003072164610974	-0.0025487836998964	0.00046548936507400003	0.001897782628966
0.010790485020034001	-0.04761088860380001	0.00635232717704	0.0011343309373408	0.003072164610974	1.6025244361189812	-0.007881618469486	0.00

In [93]:
S = np.array([[0.00000000E+00,   -0.55888279E-02,    0.37646671E-01,   -0.10215025E-01,   -0.19008043E-01,    0.10790483E-01,    0.39488698E-01,    0.22832203E-02,    0.98322250E-03],
              [-0.55888279E-02,    0.16220877E+01,   -0.70191566E-02 ,  -0.19321864E-01,   -0.16458576E-01,   -0.47610882E-01,    0.96167614E-02,   -0.44564699E-02,   -0.70201826E-04],
              [ 0.37646671E-01,   -0.70191566E-02,    0.23642283E+01 ,  -0.51777846E-02,    0.26646389E-02,    0.63523264E-02,    0.14704368E-01,    0.10976565E-02,   -0.75663869E-02],
              [-0.10215025E-01,   -0.19321864E-01,   -0.51777846E-02,    0.26022498E+01,   -0.27001494E-03,    0.11343308E-02,    0.61214594E-03,   -0.79021615E-03,   -0.36715896E-03],
              [-0.19008043E-01,   -0.16458576E-01,    0.26646389E-02 ,  -0.27001494E-03,    0.29895340E+01,    0.30721643E-02,   -0.25487833E-02,    0.46548931E-03,    0.18977824E-02],
              [ 0.10790483E-01,   -0.47610882E-01,    0.63523264E-02,    0.11343308E-02,    0.30721643E-02,    0.16025242E+01,   -0.78816174E-02,    0.35561877E-02,   -0.24620239E-01],
              [ 0.39488698E-01,    0.96167614E-02,    0.14704368E-01,    0.61214594E-03,   -0.25487833E-02,   -0.78816174E-02,    0.23984050E+01,    0.71342017E-02,   -0.15897952E-02],
              [ 0.22832203E-02,   -0.44564699E-02,    0.10976565E-02 ,  -0.79021615E-03,    0.46548931E-03,    0.35561877E-02,    0.71342017E-02,    0.25332683E+01,   -0.29516780E-02],
              [ 0.98322250E-03,   -0.70201826E-04,   -0.75663869E-02 ,  -0.36715896E-03,    0.18977824E-02,   -0.24620239E-01,   -0.15897952E-02,   -0.29516780E-02,    0.29255756E+01]
 ]
 )

In [94]:
0.000000000  -0.0055888279     0.037646671       0.010790483     0.0394887  
-0.0055888279   1.6220877         -0.00701916    -0.047610882    0.0096167614  
0.037646671     -0.0070191566     2.3642283         0.00635233      0.014704368  
0.010790483     -0.047610882      0.00635233        1.6025242       -0.00788162  
0.039488698     0.0096167614      0.014704368       -0.00788162   2.3984050  


SyntaxError: invalid syntax (1138233119.py, line 1)

In [95]:
print('\n'.join(['\t'.join(['{:4}'.format(item) for item in row]) 
      for row in np.round(S - F_del*27.2114, 7)]))

 0.0	 0.0	-0.0	 0.0	 0.0	-0.0	-0.0	-0.0	-0.0
 0.0	-3e-07	 0.0	 0.0	 0.0	 0.0	-0.0	 0.0	 0.0
-0.0	 0.0	-3e-07	 0.0	-0.0	-0.0	-0.0	-0.0	 0.0
 0.0	 0.0	 0.0	-4e-07	 0.0	-0.0	-0.0	 0.0	 0.0
 0.0	 0.0	-0.0	 0.0	-4e-07	-0.0	 0.0	-0.0	-0.0
-0.0	 0.0	-0.0	-0.0	-0.0	-2e-07	 0.0	-0.0	 0.0
-0.0	-0.0	-0.0	-0.0	 0.0	 0.0	-4e-07	-0.0	 0.0
-0.0	 0.0	-0.0	 0.0	-0.0	-0.0	-0.0	-4e-07	 0.0
-0.0	 0.0	 0.0	 0.0	-0.0	 0.0	 0.0	 0.0	-5e-07
